#  Prédiction de la Vitesse Radiale Stellaires avec Incertitude (Gaia DR3)

Ce notebook implémente une approche deep learning pour estimer la vitesse radiale des étoiles à partir des données astrométriques de Gaia. 

**Points clés :**
* **Collecte sécurisée** : Utilisation de requêtes ADQL avec gestion d'erreurs.
* **Feature Engineering Physique** : Transformation des données brutes en variables interprétables (Magnitude absolue, distance).
* **Estimation de l'Incertitude** : Utilisation du **Monte Carlo Dropout** pour que l'IA quantifie son propre doute.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from astroquery.gaia import Gaia
import logging

# Configuration du suivi (Logging)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

## 1. Collecte des données
Nous récupérons 100 000 étoiles disposant de mesures complètes.

In [2]:
def fetch_gaia_data(limit=100000):
    logging.info(" Connexion aux serveurs de l'ESA (Gaia DR3)...")
    
    query = f"""
    SELECT TOP {limit} 
        source_id, ra, dec, parallax, parallax_error, 
        pmra, pmra_error, pmdec, pmdec_error,
        phot_g_mean_mag, bp_rp, radial_velocity, radial_velocity_error
    FROM gaiadr3.gaia_source 
    WHERE parallax IS NOT NULL 
      AND pmra IS NOT NULL 
      AND pmdec IS NOT NULL
      AND phot_g_mean_mag IS NOT NULL
      AND radial_velocity IS NOT NULL
    """
    
    try:
        job = Gaia.launch_job(query)
        df = job.get_results().to_pandas()
        logging.info(f" Téléchargement terminé : {len(df)} étoiles récupérées.")
        return df
    except Exception as e:
        logging.error(f" Erreur lors de la requête : {e}")
        return None

df_raw = fetch_gaia_data()

# Sauvegarde de sécurité
os.makedirs("data", exist_ok=True)
df_raw.to_parquet("data/gaia_rv_train_100k.parquet", index=False)

## 2. Feature Engineering
On transforme les mesures angulaires en propriétés physiques (Distance, Magnitude absolue).

In [3]:
def prepare_features(df):
    logging.info(" Raffinement des données stellaires...")
    
    # On filtre les parallaxes aberrantes
    df = df[df['parallax'] > 0].copy()
    
    # Calculs astrophysiques
    df['distance_pc'] = 1000 / df['parallax']
    df['absolute_mag'] = df['phot_g_mean_mag'] - 5 * np.log10(df['distance_pc']) + 5
    df['pm_total'] = np.sqrt(df['pmra']**2 + df['pmdec']**2)

    features_cols = [
        'parallax', 'pmra', 'pmdec', 'phot_g_mean_mag', 'bp_rp', 
        'distance_pc', 'absolute_mag', 'pm_total'
    ]
    
    df = df.dropna(subset=features_cols + ['radial_velocity'])
    return df[features_cols], df['radial_velocity']

X_raw, y_raw = prepare_features(df_raw)

## 3. Architecture
Ce modèle utilise le **Dropout** même pendant la prédiction pour estimer son incertitude.

In [4]:
class StellarConfidenceNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(p=0.2), # Actif lors du MC Dropout
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

model = StellarConfidenceNet()
logging.info(" Modèle prêt pour l'apprentissage.")

## 4. Entraînement : Le Coach Patient

In [5]:
# Préparation des données
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_raw, y_raw, test_size=0.2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_p)
X_test_scaled = scaler.transform(X_test_p)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), 
                                        torch.tensor(y_train_p.values, dtype=torch.float32).view(-1,1)), 
                          batch_size=256, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.MSELoss()

logging.info(f" Entraînement sur {device}...")
for epoch in range(30):
    model.train()
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Époque {epoch+1} terminée.")

## 5. Visualisation
On demande 100 fois l'avis du modèle pour chaque étoile afin de calculer une barre d'erreur.

In [6]:
def plot_uncertainty(model, X_test_scaled, y_test_p, n_stars=50):
    model.train() # Indispensable pour garder le Dropout actif
    X_sample = torch.tensor(X_test_scaled[:n_stars], dtype=torch.float32).to(device)
    
    iterations = 100
    predictions = np.array([model(X_sample).detach().cpu().numpy() for _ in range(iterations)])
    
    mean = predictions.mean(axis=0).flatten()
    std = predictions.std(axis=0).flatten()
    
    plt.figure(figsize=(12, 6))
    plt.errorbar(range(n_stars), mean, yerr=std*2, fmt='o', color='royalblue', 
                 ecolor='lightblue', capsize=4, label='IA (Moyenne ± 2σ)')
    plt.scatter(range(n_stars), y_test_p[:n_stars], color='red', marker='x', label='Vrai (Gaia)')
    
    plt.title("Doute de l'IA sur la Vitesse Radiale")
    plt.ylabel("Vitesse (km/s)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_uncertainty(model, X_test_scaled, y_test_p.values)